# BM25 단독: 512 청크에 256 청크를 더하면 나아지는가

EXP-011은 256 청크를 dense에만 더했고 BM25는 512 청크만 쓴다. BM25도 256 청크를 쓰면 BM25 단독 순위가 나아지는지 dev-full summary_q로 먼저 본다. hybrid EXP로 갈지 정하는 근거이고, EXP 판정이 아니다. 세 구성: 512만(지금), 256만, 512+256 한 색인(문서 점수는 청크 최댓값, dense와 같은 구조). BM25 설정은 지금 기준(kiwi-v1)과 같다.

In [1]:
import time
from pathlib import Path

import bm25s
import numpy as np
import pandas as pd
from kiwipiepy import Kiwi

from rag.bm25 import BM25Retriever, forms
from rag.eval.metrics import gold_ranks, load_gold
from rag.index import load_cfg
from rag.retrieve import rank_docs, read_doc_ids
from rag.search import query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
IDX = {'512': ROOT / 'data/index/snowflake-ko-fixed-512-64/full',
       '256': ROOT / 'data/index/snowflake-ko-fixed-256-32/full'}
K = 10
POOL = 200

## 질의와 정답

In [3]:
cfg = load_cfg(['+exp=exp011', 'index.scope=full'])
b = cfg.retriever.bm25
qids, text_path = query_list(cfg)
texts = read_texts(text_path, qids)
gold = load_gold(cfg, 'dev-full')
print(len(qids), 'queries', dict(b))

3016 queries {'name': 'kiwi-v1', 'tags': ['NNG', 'NNP', 'NNB', 'NR', 'NP', 'VV', 'VA', 'XR', 'SL', 'SH', 'SN'], 'method': 'lucene', 'k1': 1.5, 'b': 0.75}


## 청크 토큰화 (두 청크 파일, 공용 어휘)

In [4]:
import json

kiwi = Kiwi()
tags = set(b.tags)
vocab: dict[str, int] = {}
ids, doc_ids = {}, {}
for name, d in IDX.items():
    start = time.perf_counter()
    with (d / 'chunks.jsonl').open(encoding='utf-8') as f:
        rows = (json.loads(line)['text'] for line in f)
        ids[name] = [np.array([vocab.setdefault(w, len(vocab)) for w in doc], dtype=np.int32)
                     for doc in forms(kiwi, rows, tags)]
    doc_ids[name] = read_doc_ids(d)
    print(name, len(ids[name]), 'chunks', round(time.perf_counter() - start), 's')
print('vocab', len(vocab))

512 118039 chunks 684 s
256 220645 chunks 614 s
vocab 87314


## 세 구성의 색인

In [5]:
def index(parts):
    model = bm25s.BM25(method=b.method, k1=b.k1, b=b.b)
    model.index(([x for p in parts for x in ids[p]], vocab), show_progress=False)
    return model, np.concatenate([doc_ids[p] for p in parts])


configs = {'512': ['512'], '256': ['256'], '512+256': ['512', '256']}
models = {name: index(parts) for name, parts in configs.items()}
print({n: len(m[1]) for n, m in models.items()})

{'512': 118039, '256': 220645, '512+256': 338684}


512 구성이 지금 기준의 BM25 인덱스(`bm25-kiwi-v1`)와 같은 순위를 내는지 앞 50개 질의로 확인한다.

In [6]:
ref = BM25Retriever(cfg, IDX['512'])
ref_ids = read_doc_ids(IDX['512'])
q_tok = [[vocab[w] for w in doc if w in vocab] for doc in forms(kiwi, texts, tags)]
ref_tok = ref.query_ids(texts[:50])
m512, d512 = models['512']
same = sum(rank_docs(ref.scores(ref_tok[i]), ref_ids, K, POOL)
           == rank_docs(m512.get_scores_from_ids(q_tok[i]), d512, K, POOL) for i in range(50))
print('기준과 같은 순위', same, '/ 50')

기준과 같은 순위 50 / 50


## BM25 단독 지표 (dev-full)

In [7]:
ranks = {}
for name, (model, dids) in models.items():
    start = time.perf_counter()
    rows = [{'qid': q, 'rank': r, 'doc_id': d}
            for q, tok in zip(qids, q_tok, strict=True)
            for r, d in enumerate(rank_docs(model.get_scores_from_ids(tok), dids, K, POOL), 1)]
    ranks[name] = gold_ranks(pd.DataFrame(rows), gold, K)
    print(name, round((time.perf_counter() - start) / len(qids) * 1000, 1), 'ms/질의')
table = pd.DataFrame({n: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                      for n, r in ranks.items()})
print(table.round(4))

512 2.1 ms/질의
256 3.5 ms/질의
512+256 6.4 ms/질의
              512     256  512+256
recall@1   0.8163  0.8163   0.8352
recall@5   0.9174  0.9148   0.9214
recall@10  0.9416  0.9420   0.9413


In [8]:
base = ranks['512']
for name in ['256', '512+256']:
    c = ranks[name]
    for k in (1, 5):
        gain = int(((base.isna() | (base > k)) & (c <= k)).sum())
        loss = int(((base <= k) & (c.isna() | (c > k))).sum())
        print(f'{name} vs 512 top{k}: 진입 {gain}, 이탈 {loss}, 순증 {gain - loss}')

256 vs 512 top1: 진입 113, 이탈 113, 순증 0
256 vs 512 top5: 진입 45, 이탈 53, 순증 -8
512+256 vs 512 top1: 진입 100, 이탈 43, 순증 57
512+256 vs 512 top5: 진입 41, 이탈 29, 순증 12
